# KG handoff: validation and exploration

Use this notebook first after cloning `main`. It runs the standalone validator, inspects the current graph, and shows a real product-to-ingredient-to-concept path. It reads the delivered files without changing data or requiring Neo4j.

The cells use only Python's standard library. Open with a Python 3 Jupyter kernel or a notebook editor; a notebook interface is a separate installation from the standard-library validator. No development virtual environment is supplied.

Steps: (1) locate the graph, (2) validate evidence and structure, (3) inspect types/counts, (4) inspect a stored path, (5) inspect pending claim status. Structural validation is not clinical approval.


In [1]:
from pathlib import Path
import csv, json, hashlib
csv.field_size_limit(10_000_000)
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'reports/kg_handoff_file_list.json').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the cloned repository or its notebooks folder.')
inventory = json.loads((ROOT / 'reports/kg_handoff_file_list.json').read_text())
GRAPH = ROOT / inventory['graph']
manifest = json.loads((GRAPH / 'manifest.json').read_text())
def node_rows(kind):
    with (GRAPH / 'nodes' / (kind + '.csv')).open() as handle:
        rows = list(csv.DictReader(handle))
    for row in rows:
        row['properties'] = json.loads(row.pop('properties_json'))
        row['evidence_ids'] = json.loads(row['evidence_ids'])
    return rows
print('Graph:', GRAPH.name)


Graph: full-graph-concepts-v1-08c99550a0fa0f86aca9


In [2]:
import importlib.util
spec = importlib.util.spec_from_file_location('handoff_validation', ROOT / 'scripts/validate_kg_handoff.py')
validator = importlib.util.module_from_spec(spec)
spec.loader.exec_module(validator)
validation = validator.validate(ROOT)
print(json.dumps(validation, indent=2))


{
  "status": "passed",
  "graph": "data/graph/full-graph-concepts-v1-08c99550a0fa0f86aca9",
  "package_files": 484,
  "nodes": 35998,
  "relationships": 74122,
  "evidence_pointers": 64312,
  "pending_claims": 1269,
  "field_support_anchors": 2923,
  "scope": "Consumption package integrity and source support; not clinical approval or full rebuild validation"
}


In [3]:
print('Node types:')
for kind, count in sorted(manifest['node_types'].items()):
    print(f'  {kind}: {count:,}')
print('Relationship types:')
for kind, count in sorted(manifest['edge_types'].items()):
    print(f'  {kind}: {count:,}')
print('Schema details: schemas/current_kg_reference.json')


Node types:
  ChemicalCompound: 239
  ClinicalClaim: 1,269
  ExternalRecordReference: 1,277
  IngredientOccurrence: 5,438
  LabelAssertion: 11,954
  Monograph: 10
  MonographSubject: 10
  ProductLabel: 400
  RedditComment: 4,593
  RedditPost: 287
  ReferenceDocument: 10
  Serving: 400
  SourceExcerpt: 1,204
  SourcePassage: 6,250
  SupplementTopic: 10
  TerminologyConcept: 2,647
Relationship types:
  ABOUT_OCCURRENCE: 5,491
  ABOUT_SUBJECT: 1,269
  DECLARES_OCCURRENCE: 5,438
  DECLARES_SERVING: 400
  DENOTES_CONCEPT: 2,711
  EXACT_MATERIAL_MATCH: 7
  EXCERPT_OF: 1,269
  HAS_ASSERTION: 11,954
  HAS_DECLARED_COMPONENT: 786
  HAS_DECLARED_FORM: 1,371
  HAS_DECLARED_TOPIC: 514
  HAS_PENDING_CLAIM: 1,269
  HAS_REFERENCE_DOCUMENT: 10
  HAS_SERVING_BASIS: 2,392
  HAS_SOURCE_TOPIC: 10
  HAS_STANDARDIZATION_MARKER: 4
  HAS_SUBJECT: 10
  INGREDIENT_NAME_MENTIONS_TOPIC: 960
  INHERITED_THREAD_TOPIC: 6,310
  IN_THREAD: 4,820
  IN_THREAD_REFERENCE: 60
  LABEL_TITLE_MENTIONS_TOPIC: 396
  MEMBER_OF_C

In [4]:
# A declared product ingredient and its accepted general-concept link.
label_id = '210829'
labels = node_rows('ProductLabel')
label = next(n for n in labels if n['properties']['source_record']['native_id'] == label_id)
occurrences = {n['id']: n for n in node_rows('IngredientOccurrence')}
concepts = {n['id']: n for n in node_rows('TerminologyConcept')}
with (GRAPH / 'relationships/relationships.csv').open() as handle:
    edges = list(csv.DictReader(handle))
print('Product:', label['properties']['attributes']['fullName'])
print('Brand:', label['properties']['attributes']['brandName'], '| DSLD:', label_id)
found = 0
for edge in edges:
    if edge['start_id'] != label['id'] or edge['type'] != 'DECLARES_OCCURRENCE':
        continue
    occurrence = occurrences[edge['end_id']]
    if occurrence['properties'].get('original_name') != 'Ashwagandha extract':
        continue
    print('Ingredient notes:', occurrence['properties']['attributes'].get('notes'))
    for mapping in edges:
        if mapping['start_id'] == occurrence['id'] and mapping['type'] == 'DENOTES_CONCEPT':
            target = concepts.get(mapping['end_id'])
            if target is None:
                continue
            found += 1
            print('Concept:', target['properties']['display_name'])
            print('Source / ID:', target['properties']['source'], target['properties']['native_id'])
            print('Confidence:', mapping['confidence'])
            print('Scope:', json.loads(mapping['properties_json']).get('scope'))
assert found, 'Expected example concept link is missing.'


Product: Ashwagandha Extract
Brand: Seeking Health | DSLD: 210829
Ingredient notes: Ashwagandha extract (Form: 5% Withanolides) (Alt. Name: KSM-66) PlantPart: root Genus: Withania Species: somnifera 
Concept: Withania somnifera root extract
Source / ID: rxnorm 1309279
Confidence: High
Scope: General ingredient concept only; standardization, branded preparation and quantities remain on the source occurrence. No exact material or clinical equivalence.


In [5]:
claims = node_rows('ClinicalClaim')
example = next(n for n in claims if n['properties']['original_fields']['ingredient']['name'] == 'Creatine'
               and n['properties']['original_fields']['claim_type'] == 'dose')
p = example['properties']
f = p['original_fields']
print(json.dumps({'node_id': example['id'], 'claim_type': f['claim_type'],
                  'ingredient': f['ingredient'], 'population': f['population'],
                  'review_state': p['review_state'], 'clinical_approval': p['clinical_approval'],
                  'source_url': f['source']['url'], 'source_span': p['source_span']}, indent=2))
print('Inspect the stored excerpt for all qualifications; this is not a recommended dose.')


{
  "node_id": "claim:40b8f09332319c60632e15306d57445bf74cc96d370abb16834b417ca67cf4e7",
  "claim_type": "dose",
  "ingredient": {
    "formulation": null,
    "name": "Creatine",
    "strain": null
  },
  "population": "Adult",
  "review_state": "pending",
  "clinical_approval": false,
  "source_url": "https://naturalmedicines.therapeuticresearch.com/IngredientsTherapiesMonographs/Pro?id=873",
  "source_span": {
    "coordinate_system": "unicode_codepoints",
    "end": 241,
    "file": "data/raw/natmedpro/records.jsonl",
    "pointer": "/3/sections/7/text",
    "sha256": "b413d5ff9051e5ffdbd5c44b804aaf6b9a17f4cdf449e7f60937a7e901b4560c",
    "start": 0
  }
}
Inspect the stored excerpt for all qualifications; this is not a recommended dose.


## Next

Use `Embedding_and_Retrieval_Starter.ipynb` to inspect potential evidence text and plan the indexes. Do not merge current graph folders with older snapshots. Historical construction notebooks and full rebuild inputs remain on `preprocessing-kg-schema`.
